# 🧬 LOGO Evaluation — Kaggle (Step 7/8)
**Leave-One-Generator-Out cross-generator validation.**

## ⚙️ Setup
1. **Datasets** — Add all 12 datasets (Input → Add Data)
2. **Secret** — Add `HF_TOKEN` secret
3. **Accelerator** — GPU T4 x2
4. **Run All**

**Sequence**: Tiny → Base → Large → Baselines → Ablation → PaperEvals → **LOGO** → Outputs

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 0: Clone Repo + Install Dependencies
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os
from pathlib import Path

REPO_URL  = "https://github.com/MIHMahmudEli/ai-image-detection-research.git"
CLONE_DIR = Path("/kaggle/working/ai-image-detection-research")

if not CLONE_DIR.exists():
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
else:
    print("Pulling latest...")
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--rebase"], check=False)

os.chdir(str(CLONE_DIR))
sys.path.insert(0, str(CLONE_DIR / "model"))

subprocess.run([sys.executable, "-m", "pip", "install",
    "huggingface_hub", "open_clip_torch", "scipy", "scikit-learn",
    "-q", "--disable-pip-version-check"], check=False)

print(f"Project root: {CLONE_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1: Imports & Environment
# ═══════════════════════════════════════════════════════════
import os, sys, math, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path("/kaggle/working/ai-image-detection-research/model")))
from src.kaggle_utils import KaggleEnv
env = KaggleEnv(project_root_search=True)
PROJECT_ROOT = env.project_root
os.chdir(env.working_dir)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE_TEST = not torch.cuda.is_available()
print(f"Device: {device} | SMOKE: {SMOKE_TEST}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2: LOGO Run Setup
# ═══════════════════════════════════════════════════════════
from src.model import build_mfft
from src.dataset import create_split_dataloaders
from src.logo_eval import create_logo_manifests
from sklearn.metrics import roc_auc_score, f1_score
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR

LOGO_DIR = PROJECT_ROOT / "dataset" / "metadata" / "logo"
OUT = PROJECT_ROOT / "paper" / "result" / ("verify" if SMOKE_TEST else "full_scale") / "logo"
OUT.mkdir(parents=True, exist_ok=True)

_manifest = PROJECT_ROOT / "dataset" / "metadata" / "train_manifest.csv"
if not _manifest.exists(): env.download_manifest(_manifest)
if not _manifest.exists(): env.rebuild_manifest_from_kaggle(_manifest)
if not _manifest.exists(): _manifest = PROJECT_ROOT / "dataset" / "metadata" / "clean_metadata.csv"

create_logo_manifests(str(_manifest), LOGO_DIR)
train_files = sorted(LOGO_DIR.glob("logo_train_wo_*.csv"))
generators = [f.stem.replace("logo_train_wo_", "") for f in train_files]
print(f"Found {len(generators)} LOGO groups: {generators}")

if SMOKE_TEST: generators = generators[:1]

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3: LOGO Training & Evaluation
# ═══════════════════════════════════════════════════════════
results = []
res_file = OUT / "logo_results.csv"
if res_file.exists():
    print("Loading existing results...")
    results = pd.read_csv(res_file).to_dict('records')
    done_gens = [r['generator'] for r in results]
else:
    done_gens = []

IMAGE_SIZE, BATCH, NUM_EPOCHS = (224, 8, 1) if SMOKE_TEST else (384, 64, 10)
AMP = torch.cuda.is_available()

for gen in generators:
    if gen in done_gens:
        print(f"Skipping {gen} (already done)")
        continue

    print(f"\n{'='*50}\nTraining without {gen} (LOGO)\n{'='*50}")
    tr_csv = LOGO_DIR / f"logo_train_wo_{gen}.csv"
    te_csv = LOGO_DIR / f"logo_test_{gen}.csv"

    loader_tr, _, _ = create_split_dataloaders(
        str(PROJECT_ROOT), [str(tr_csv)], batch_size=BATCH, num_workers=0 if SMOKE_TEST else 4,
        size=IMAGE_SIZE, val_split=0.0, test_split=0.0, use_weighted_sampler=True,
        max_samples=600 if SMOKE_TEST else None
    )
    _, _, loader_te = create_split_dataloaders(
        str(PROJECT_ROOT), [str(te_csv)], batch_size=BATCH, num_workers=0 if SMOKE_TEST else 4,
        size=IMAGE_SIZE, val_split=0.0, test_split=1.0,
        max_samples=200 if SMOKE_TEST else None
    )

    model = build_mfft("base").to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = SequentialLR(optimizer, [
        LinearLR(optimizer, 0.01, 1.0, total_iters=min(500, len(loader_tr))),
        CosineAnnealingWarmRestarts(optimizer, NUM_EPOCHS*len(loader_tr))
    ], milestones=[min(500, len(loader_tr))])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    criterion = nn.CrossEntropyLoss()

    # Train
    for epoch in range(NUM_EPOCHS):
        model.train(); total_loss = correct = total = 0
        for x, y in tqdm(loader_tr, desc=f"{gen} Ep {epoch+1}", leave=False):
            x, y = x.to(device), y.to(device)
            with torch.amp.autocast("cuda", enabled=AMP):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(); scheduler.step()
    
    # Evaluate
    model.eval(); y_true, y_prob = [], []
    with torch.no_grad():
        for x, y in tqdm(loader_te, desc=f"{gen} Test"):
            try:
                y_prob.extend(F.softmax(model(x.to(device)), dim=-1)[:,1].cpu().numpy())
                y_true.extend(y.numpy())
            except: pass
    
    yt, yp = np.array(y_true), np.array(y_prob)
    yp_cls = (yp >= 0.5).astype(int)
    acc = (yp_cls == yt).mean() * 100
    auc = roc_auc_score(yt, yp) if len(np.unique(yt)) > 1 else 0
    f1  = f1_score(yt, yp_cls, zero_division=0) * 100
    
    print(f"Held-out {gen}: Acc={acc:.2f}%, AUC={auc:.4f}, F1={f1:.2f}%")
    results.append({"generator": gen, "acc": round(acc,2), "auc": round(auc,4), "f1": round(f1,2)})
    
    pd.DataFrame(results).to_csv(res_file, index=False)
    env.upload_to_hf(res_file, env.hf_results_repo, f"results/logo/logo_results.csv")

print("\n=== LOGO SUMMARY ===")
df = pd.DataFrame(results)
print(df)
print(f"\nMean Accuracy: {df['acc'].mean():.2f}%")
print("\n✅ LOGO Eval complete. Proceed to 08_generate_outputs.ipynb")